In [ ]:
# ---- Cell 1: Install and initialize ----
!pip install -q earthengine-api geemap geopandas scikit-learn

import ee, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

ee.Authenticate()          # skip if already authenticated in this session
PROJECT_ID = 'terraworks-511519'
ee.Initialize(project=PROJECT_ID)
print('Earth Engine ready')

# ---- Cell 2: Settings ----
UJANI_LAT, UJANI_LON = 18.0, 75.1
centre = ee.Geometry.Point([UJANI_LON, UJANI_LAT])
aoi = centre.buffer(10000).bounds()

# Single dry-season composite used for classification
COMPOSITE_START, COMPOSITE_END = '2023-11-01', '2024-03-31'
SAMPLES_PER_CLASS = 200
TRAIN_FRACTION = 0.7
RANDOM_SEED = 42

WATER_CSV = 'ujani_water_extent_monthly.csv'     # output of notebook 02
RAIN_CHECKPOINT = 'ujani_rainfall_checkpoint.csv'

# ---- Cell 3: Build the feature image (one dry-season composite) ----
def mask_s2(img):
    scl = img.select('SCL')
    clear = (scl.neq(0).And(scl.neq(3)).And(scl.neq(8))
                 .And(scl.neq(9)).And(scl.neq(10)))
    return img.updateMask(clear).divide(10000).copyProperties(img, ['system:time_start'])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate(COMPOSITE_START, COMPOSITE_END)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(mask_s2))

composite = s2.median().clip(aoi)
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('ndwi')
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('ndvi')
features = composite.select(['B2', 'B3', 'B4', 'B8', 'B11']).addBands([ndwi, ndvi])
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'ndwi', 'ndvi']

# ---- Cell 4: Label image (three classes) ----
# 0 = water, 1 = vegetation, 2 = bare or built land
label = (ee.Image(0)
         .where(ndwi.gt(0.2).And(ndvi.lt(0)), 0)
         .where(ndvi.gt(0.4), 1)
         .where(ndvi.lt(0.2).And(ndwi.lt(0)), 2)
         .rename('label')
         .updateMask(ndwi.gt(0.2).And(ndvi.lt(0))
                     .Or(ndvi.gt(0.4))
                     .Or(ndvi.lt(0.2).And(ndwi.lt(0))))  # keep only confident pixels
         .clip(aoi))
# ---- Cell 5: Stratified sample points (one server call) ----
samples = features.addBands(label).stratifiedSample(
    numPoints=SAMPLES_PER_CLASS,
    classBand='label',
    region=aoi,
    scale=10,
    seed=RANDOM_SEED,
    geometries=True
).randomColumn('rand', RANDOM_SEED)

train = samples.filter(ee.Filter.lt('rand', TRAIN_FRACTION))
test = samples.filter(ee.Filter.gte('rand', TRAIN_FRACTION))

print('Training points:', train.size().getInfo())
print('Test points:', test.size().getInfo())

# ---- Cell 6: Train and evaluate the Random Forest ----
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=RANDOM_SEED) \
    .train(features=train, classProperty='label', inputProperties=BANDS)

test_classified = test.classify(classifier, 'predicted')
matrix = test_classified.errorMatrix('label', 'predicted')

print('Confusion matrix (rows = actual, columns = predicted; 0 water, 1 vegetation, 2 bare/built):')
print(np.array(matrix.array().getInfo()))
print(f"Overall accuracy: {matrix.accuracy().getInfo():.3f}")
print(f"Kappa:            {matrix.kappa().getInfo():.3f}")


# ---- Cell 7: Classify the image and view it ----
# FIX: Use 'features' instead of 'composite' because 'composite' does not contain 'ndwi' and 'ndvi'
classified = features.classify(classifier, 'class')

import geemap
palette = ['1f78b4', '33a02c', 'b15928']   # water, vegetation, bare/built
Map = geemap.Map(center=[UJANI_LAT, UJANI_LON], zoom=12)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'True colour')
Map.addLayer(classified, {'min': 0, 'max': 2, 'palette': palette}, 'Classified (0 water, 1 veg, 2 bare)')
Map

# ---- Cell 8: Load monthly water extent from notebook 02 ----
water = pd.read_csv(WATER_CSV)
water['date'] = pd.to_datetime(water['month'])
water = water.sort_values('date').reset_index(drop=True)

# Use Sentinel-2 where available, otherwise Sentinel-1 (radar fills cloudy months)
water['water_km2'] = water['s2_water_km2'].fillna(water['s1_water_km2'])
water['source'] = np.where(water['s2_water_km2'].notna(), 'S2', np.where(water['s1_water_km2'].notna(), 'S1', 'none'))
water = water.dropna(subset=['water_km2']).reset_index(drop=True)
print(f'{len(water)} usable months, sources: {water["source"].value_counts().to_dict()}')

# ---- Cell 9: Monthly rainfall from CHIRPS (one request per month, with checkpointing) ----
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(aoi)

def rain_month(month_str):
    start = ee.Date(month_str)
    total = (chirps.filterDate(start, start.advance(1, 'month'))
             .select('precipitation').sum()
             .reduceRegion(ee.Reducer.mean(), aoi, 5566, maxPixels=1e9,
                           bestEffort=True, tileScale=4)
             .get('precipitation'))
    return ee.Number(total).getInfo()

def call_with_retry(fn, arg, retries=5):
    for attempt in range(retries):
        try:
            return fn(arg)
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ['concurrent', 'quota', 'rate', 'too many']) and attempt < retries - 1:
                wait = 10 * (2 ** attempt)
                print(f'  limit reached, waiting {wait}s...')
                time.sleep(wait)
            else:
                raise

if os.path.exists(RAIN_CHECKPOINT):
    rain_df = pd.read_csv(RAIN_CHECKPOINT)
else:
    rain_df = pd.DataFrame(columns=['month', 'rain_mm'])

done = set(rain_df['month'])
rows = rain_df.to_dict('records')
for m in water['date'].dt.strftime('%Y-%m-%d'):
    month = m[:7]
    if month in done:
        continue
    rows.append({'month': month, 'rain_mm': call_with_retry(rain_month, m)})
    pd.DataFrame(rows).to_csv(RAIN_CHECKPOINT, index=False)
    time.sleep(1)

rain_df = pd.DataFrame(rows)
rain_df['date'] = pd.to_datetime(rain_df['month'])

# ---- Cell 10: Build forecast features ----
data = water.merge(rain_df[['month', 'rain_mm']], on='month', how='inner')
data = data.sort_values('date').reset_index(drop=True)

data['rain_lag1'] = data['rain_mm'].shift(1)
data['water_lag1'] = data['water_km2'].shift(1)
data['month_sin'] = np.sin(2 * np.pi * data['date'].dt.month / 12)
data['month_cos'] = np.cos(2 * np.pi * data['date'].dt.month / 12)
data = data.dropna().reset_index(drop=True)

FEATURES = ['rain_mm', 'rain_lag1', 'water_lag1', 'month_sin', 'month_cos']
print(f'{len(data)} rows available for the forecast')

# ---- Cell 11: Train, test, and compare against a simple baseline ----
split = int(len(data) * 0.8)
train_f, test_f = data.iloc[:split], data.iloc[split:]

model = Ridge(alpha=1.0).fit(train_f[FEATURES], train_f['water_km2'])
pred = model.predict(test_f[FEATURES])

mae_model = mean_absolute_error(test_f['water_km2'], pred)
mae_naive = mean_absolute_error(test_f['water_km2'], test_f['water_lag1'])  # "same as last month"

print(f'Test months: {len(test_f)}')
print(f'Model MAE:          {mae_model:.2f} km²')
print(f'Naive baseline MAE: {mae_naive:.2f} km²')
if mae_model < mae_naive:
    print('The model beats the naive baseline on held-out months.')
else:
    print('The model does not beat the naive baseline. Do not present this forecast as useful.')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_f['date'], test_f['water_km2'], 'o-', label='Observed')
ax.plot(test_f['date'], pred, 's--', label='Model')
ax.set_title('Held-out test period: observed vs predicted water area')
ax.set_ylabel('Water area (km²)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ---- Cell 12: Forecast the next 6 months (rainfall assumed at historical average) ----
# Future rainfall is unknown, so each future month uses its long-run average.
# Treat this as a scenario, not a prediction.
clim = data.groupby(data['date'].dt.month)['rain_mm'].mean()

last = data.iloc[-1].copy()
future_rows = []
prev_water = last['water_km2']
prev_rain = last['rain_mm']
future_dates = pd.date_range(last['date'] + pd.DateOffset(months=1), periods=6, freq='MS')

for d in future_dates:
    rain = clim[d.month]
    row = {
        'date': d,
        'rain_mm': rain,
        'rain_lag1': prev_rain,
        'water_lag1': prev_water,
        'month_sin': np.sin(2 * np.pi * d.month / 12),
        'month_cos': np.cos(2 * np.pi * d.month / 12),
    }
    row['water_km2_forecast'] = max(model.predict(pd.DataFrame([row])[FEATURES])[0], 0)
    future_rows.append(row)
    prev_water = row['water_km2_forecast']
    prev_rain = rain

forecast = pd.DataFrame(future_rows)[['date', 'water_km2_forecast']]
print(forecast.to_string(index=False))
print('\nForecast uses historical average rainfall for future months. Error is about the model MAE above.')

# ---- Cell 13: Save outputs ----
data.to_csv('ujani_model_dataset.csv', index=False)
forecast.to_csv('ujani_forecast_6month.csv', index=False)
print('Saved ujani_model_dataset.csv and ujani_forecast_6month.csv')